# BƯỚC 4: ĐÁNH GIÁ PHÂN CỤM, HỒ SƠ PERSONA & SUY LUẬN MỚI (EVALUATION & PROFILING)

## Mục tiêu theo đề bài Practice Exercise 3:
1. Đánh giá chất lượng phân cụm qua Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz Score và Dunn Index.
2. Trực quan hóa không gian 2D, 3D và biểu đồ Silhouette Knife Plot.
3. Xây dựng bảng hồ sơ khách hàng (`cluster_profiles.csv`) và định danh Persona động cùng chiến lược Marketing.
4. Lưu gói mô hình và kiểm chứng dự đoán trên tập Holdout Test (20%).

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import METRICS_DIR, PROCESSED_DATA_DIR
from src.deployment.predictor import CustomerSegmentPredictor
from src.evaluation.profiling import create_cluster_profiles, generate_customer_personas
from src.evaluation.visualization import plot_clusters_2d, plot_clusters_3d, plot_silhouette_analysis
from src.models.persistence import load_model

train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
test_df = pd.read_csv(PROCESSED_DATA_DIR / "test.csv")
comparison_df = pd.read_csv(METRICS_DIR / "model_comparison.csv")
comparison_df

In [ ]:
# Nạp lại mô hình K-Means tốt nhất đã lưu để vẽ biểu đồ & lập chân dung phân khúc
bundle = load_model("final_kmeans_pipeline.pkl")
model = bundle["model"]
scaler = bundle["scaler"]

ma_tran_train = scaler.transform(train_df)
nhan_train = model.predict(ma_tran_train)
tam_cum_goc = scaler.inverse_transform(model.cluster_centers_)

plot_clusters_2d(train_df, nhan_train, centroids=tam_cum_goc)
plot_clusters_3d(train_df, nhan_train)
plot_silhouette_analysis(ma_tran_train, nhan_train, optimal_k=model.n_clusters)

In [ ]:
# Lập bảng hồ sơ thống kê và Persona kinh doanh
profile_df = create_cluster_profiles(train_df, nhan_train)
personas = generate_customer_personas(profile_df)
print(profile_df.to_string(index=False))
pd.DataFrame(personas)

In [ ]:
# Chạy thử nghiệm suy luận trên 1 khách hàng mới và toàn bộ tập Holdout Test (20%)
predictor = CustomerSegmentPredictor("final_kmeans_pipeline.pkl")
print(predictor.predict_single_customer(age=35, annual_income=75000.0, total_spending=1400.0))
test_preds = predictor.predict_batch(test_df)
test_preds[["ID", "Age", "Income", "Total_Spending", "Predicted_Cluster", "Persona_Name"]].head(10)